# Semana 7: Apuntes de la clase

**Teoría moderna de portafolios: Markowitz y la frontera eficiente** (CFA L1, Portfolio Management: *Portfolio Risk and Return: Part I y Part II*; CFA L2: *Backtesting and Simulation*)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/07_markowitz_frontera/clase07_apuntes.ipynb)

Este notebook resume los conceptos que debiste llevarte de la clase, con los ejemplos numéricos ejecutables. Es la última clase antes del examen parcial (miércoles 21/10, semanas 1 a 7): la sección final es un mapa de repaso de todo lo visto.

## Glosario de siglas de la semana

Antes de los conceptos, el idioma. Estas son las siglas y símbolos que usamos esta semana; los de origen inglés se usan tal cual en la práctica profesional y en el examen CFA.

| Sigla | Significado | En pocas palabras |
|---|---|---|
| MPT | *Modern Portfolio Theory* (teoría moderna de portafolios) | El marco de Markowitz (1952): elegir portafolios mirando solo su media y su varianza. |
| $w$, $\mu$, $\Sigma$ | Pesos, retornos esperados y matriz de covarianzas | Los tres ingredientes del problema; $\sigma_p^2 = w^{\top}\Sigma\,w$ es la fórmula de la semana 6 escrita para $n$ activos. |
| PMV | Portafolio de mínima varianza global | La punta izquierda de la frontera: el menor riesgo alcanzable. Solo necesita $\Sigma$. |
| Frontera eficiente | *Efficient frontier* | Los portafolios no dominados: la rama superior de la frontera de mínima varianza, del PMV hacia arriba. |
| $R_f$ | *Risk-free rate* (tasa libre de riesgo) | Rinde sin varianza ni covarianza; debe estar en la misma moneda y frecuencia que los retornos. |
| CAL | *Capital Allocation Line* (línea de asignación de capital) | Las mezclas de $R_f$ con un portafolio riesgoso: una recta que nace en $R_f$. |
| Sharpe | Ratio de Sharpe, $(E(R_p) - R_f)/\sigma_p$ | Retorno en exceso por punto de riesgo: la pendiente de la CAL. |
| $T$ | Portafolio tangente | El portafolio riesgoso de máximo Sharpe; su CAL es tangente a la frontera eficiente. |
| $y$ y $A$ | Fracción invertida en $T$ y coeficiente de aversión al riesgo | $y^{*} = (E(R_T) - R_f)/(A\sigma_T^2)$; con $y > 1$ hay apalancamiento. |
| CML | *Capital Market Line* (línea del mercado de capitales) | La CAL cuando el tangente es el portafolio de mercado; puerta de entrada al CAPM (semana 9). |
| SLSQP | *Sequential Least Squares Programming* | El algoritmo de `scipy.optimize` que resuelve el problema con restricciones. |

## 1. El problema media-varianza

Para cada retorno objetivo $\mu^{*}$, Markowitz busca los pesos de menor riesgo:

$$\min_{w}\; w^{\top}\Sigma\,w \qquad \text{sujeto a} \qquad w^{\top}\mu = \mu^{*}, \qquad \sum_i w_i = 1 \qquad (w_i \geq 0 \text{ sin ventas en corto})$$

El costo de entrada son los insumos: $n$ retornos esperados, $n$ varianzas y $n(n-1)/2$ covarianzas. Con 14 activos son 119 parámetros estimados.

Seguimos con los activos de la semana 6 y sumamos un banco y la tasa libre de riesgo de la semana 2.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
from utils.finanzas import (min_varianza, portafolio_tangente, frontera_eficiente,
                            portafolio_dos_activos, riesgo_portafolio)

nombres = ["minera", "electrica", "banco"]
mu = pd.Series([12.0, 8.0, 10.0], index=nombres)               # retornos esperados, en %
sigma = np.array([24.0, 12.0, 18.0])                            # volatilidades, en %
corr = np.array([[1.00, 0.25, 0.40],
                 [0.25, 1.00, 0.30],
                 [0.40, 0.30, 1.00]])
cov = pd.DataFrame(np.outer(sigma, sigma) * corr, index=nombres, columns=nombres)
rf = 4.5

n = 14
print(f"Insumos para {n} activos: {n} medias + {n} varianzas + {n * (n - 1) // 2} covarianzas = {2 * n + n * (n - 1) // 2}")
cov

## 2. La frontera eficiente y el portafolio de mínima varianza global

Resolver el problema para cada $\mu^{*}$ traza la frontera de mínima varianza. La **frontera eficiente** es solo su rama superior: un portafolio es eficiente si ningún otro da más retorno con el mismo riesgo ni menos riesgo con el mismo retorno. Lo demás está dominado.

El **PMV** es la punta izquierda. Su propiedad especial: solo usa $\Sigma$, no los retornos esperados.

In [ ]:
def describir(w, etiqueta):
    w = pd.Series(w, index=mu.index[:len(w)])
    e, s = w @ mu[w.index], riesgo_portafolio(w, cov.loc[w.index, w.index])
    pesos = " / ".join(f"{x:.1%}" for x in w)
    print(f"{etiqueta:32s} pesos {pesos:24s} E(R) = {e:5.2f}%  sigma = {s:5.2f}%  Sharpe = {(e - rf) / s:.3f}")

describir(min_varianza(cov.iloc[:2, :2]), "PMV con minera y electrica")
describir(min_varianza(cov), "PMV con los tres activos")

print("\nAlgunos puntos de la frontera eficiente de tres activos:")
frontera_eficiente(mu, cov, n_puntos=6).round(2)

Salida esperada: con dos activos el PMV es 12.5% / 87.5% con $\sigma = 11.62\%$ (el de la semana 6); con tres activos pasa a 6.2% / 73.7% / 20.1%, con $E(R) = 8.65\%$ y $\sigma = 11.12\%$. Sumar el banco bajó el riesgo y subió el retorno del PMV, aunque el banco no es el mejor activo en nada: aporta por sus correlaciones. Más activos nunca empeoran el menú.

## 3. El activo libre de riesgo, la CAL y el portafolio tangente

Si pongo una fracción $y$ en un portafolio riesgoso $P$ y el resto en $R_f$:

$$E(R_c) = R_f + y\,[E(R_P) - R_f] \qquad \sigma_c = y\,\sigma_P \qquad\Rightarrow\qquad E(R_c) = R_f + \frac{E(R_P) - R_f}{\sigma_P}\,\sigma_c$$

Es una recta que nace en $R_f$: la **CAL**. Su pendiente es el **ratio de Sharpe** de $P$. Como todas las CAL nacen en el mismo punto, la mejor es la más empinada, y el portafolio que la genera es el **tangente**: el de máximo Sharpe.

In [ ]:
dos = ["minera", "electrica"]
print("Con minera y electrica (rf = 4.5%):")
for etiqueta, w in [("Minera sola", [1, 0]), ("Electrica sola", [0, 1]),
                    ("Minima varianza", min_varianza(cov.loc[dos, dos])), ("Mitad y mitad", [0.5, 0.5]),
                    ("TANGENTE", portafolio_tangente(mu[dos], cov.loc[dos, dos], rf))]:
    describir(np.asarray(w, dtype=float), etiqueta)

# Verificacion con la formula cerrada de dos activos
ea, eb = mu["minera"] - rf, mu["electrica"] - rf
c_ab = cov.loc["minera", "electrica"]
w_a = (ea * sigma[1] ** 2 - eb * c_ab) / (ea * sigma[1] ** 2 + eb * sigma[0] ** 2 - (ea + eb) * c_ab)
print(f"\nFormula cerrada: peso de la minera en el tangente = {w_a:.4f}")

Salida esperada: Sharpe de 0.312 (minera), 0.292 (eléctrica), 0.344 (PMV), 0.374 (mitad y mitad) y **0.382** para el tangente, que pone 35.9% en la minera y 64.1% en la eléctrica, con $E(R_T) = 9.44\%$ y $\sigma_T = 12.91\%$. Dos lecciones: el PMV no es el tangente (minimizar riesgo no es maximizar la recompensa por riesgo), y el tangente tampoco es el activo de mayor retorno.

Para dos activos existe fórmula cerrada (con $\tilde{\mu} = \mu - R_f$):

$$w_A^{T} = \frac{\tilde{\mu}_A\sigma_B^2 - \tilde{\mu}_B\sigma_{AB}}{\tilde{\mu}_A\sigma_B^2 + \tilde{\mu}_B\sigma_A^2 - (\tilde{\mu}_A + \tilde{\mu}_B)\sigma_{AB}}$$

## 4. Cuánto riesgo tomar: la separación en dos fondos

Todos los inversionistas quieren el mismo portafolio riesgoso, $T$. La aversión al riesgo solo decide la dosis:

$$y^{*} = \frac{E(R_T) - R_f}{A\,\sigma_T^2} \qquad \text{(retornos en decimales)}$$

In [ ]:
w_t = portafolio_tangente(mu[dos], cov.loc[dos, dos], rf)
e_t, s_t = w_t @ mu[dos], riesgo_portafolio(w_t, cov.loc[dos, dos])

for A in [4, 2]:
    y = (e_t - rf) / 100 / (A * (s_t / 100) ** 2)
    e_c, s_c = rf + y * (e_t - rf), y * s_t
    U = e_c / 100 - 0.5 * A * (s_c / 100) ** 2
    print(f"A = {A}: y* = {y:.0%} en T -> E(Rc) = {e_c:.2f}%  sigma_c = {s_c:.2f}%  U = {U:.4f}")

print(f"\nSobre la CAL, con sigma = 12% (la de la electrica sola): E(R) = {rf + (e_t - rf) / s_t * 12:.2f}% (la electrica rinde 8%)")

Salida esperada: el conservador ($A = 4$) pone 74% en $T$ y 26% en $R_f$: $E(R_c) = 8.16\%$, $\sigma_c = 9.56\%$ y utilidad de 0.0633, más que el 0.0580 que lograba la semana pasada sin activo libre de riesgo. El agresivo ($A = 2$) pone 148%: se endeuda 48% a $R_f$ para invertir más en $T$. Con el mismo riesgo que la eléctrica sola (12%), la CAL paga 9.09% en lugar de 8%: **la CAL domina a la frontera**.

Dos decisiones separadas: una técnica (hallar $T$, igual para todos) y una personal (elegir $y$).

## 5. Con tres activos el tangente mejora

In [ ]:
describir(portafolio_tangente(mu[dos], cov.loc[dos, dos], rf), "Tangente con dos activos")
describir(portafolio_tangente(mu, cov, rf), "Tangente con tres activos")
print(f"Sharpe del banco solo = {(mu['banco'] - rf) / sigma[2]:.3f}")

Salida esperada: con tres activos el tangente es 24.4% / 47.6% / 28.0%, con Sharpe de **0.411** (antes 0.382). El banco solo tiene un Sharpe de 0.306, peor que el tangente anterior, y aun así agregarlo mejora el conjunto: un activo no se juzga solo, sino por lo que le aporta al portafolio.

Si todos los inversionistas compartieran las mismas expectativas, todos tendrían el mismo $T$ y ese $T$ sería el portafolio de mercado: la CAL se vuelve la CML y de ahí sale el CAPM (semana 9).

## 6. Los límites prácticos: la fragilidad de los pesos

El modelo es correcto; el problema son sus insumos. El optimizador trata a $\mu$ como si fuera cierto, y $\mu$ es justo lo que peor se estima. Mira qué pasa si el retorno esperado de la minera cambia apenas un punto:

In [ ]:
for m in [11, 12, 13]:
    mu_alt = mu.copy(); mu_alt["minera"] = m
    w = portafolio_tangente(mu_alt, cov, rf)
    print(f"E(R) de la minera = {m}% -> tangente: " + " / ".join(f"{x:.1%}" for x in w))
print("PMV (no usa mu):                   " + " / ".join(f"{x:.1%}" for x in min_varianza(cov)))

Salida esperada: el peso de la minera en el tangente pasa de 18.4% a 24.4% y a 30.3%: doce puntos de diferencia por un cambio de dos puntos en un insumo que nadie conoce con esa precisión. El PMV no se mueve, porque no usa $\mu$.

Las defensas de la práctica profesional: restricciones (sin cortos, topes por activo y sector), portafolios que no usan $\mu$ (mínima varianza, pesos iguales, paridad de riesgo), mejores insumos (*shrinkage*, retornos de equilibrio al estilo Black-Litterman, modelos de factores) y simulación (remuestrear y promediar). Markowitz no es una máquina de dar pesos: es el marco que ordena la conversación.

## 7. Respuestas a los ítems de la clase

**1. B.** Y está dominado por X (mismo retorno, menos riesgo) y por Z (mismo riesgo, más retorno). X y Z pueden ser eficientes; Y no.

**2. C.** Con préstamo y endeudamiento a $R_f$ se elige el mayor Sharpe: P $= (10-4)/12 = 0.50$; Q $= (13-4)/20 = 0.45$; R $= (7-4)/5 = 0.60$. Aunque R rinda poco, apalancado sobre su CAL le gana a los otros en cualquier nivel de riesgo.

**3. A.** El PMV solo necesita la matriz de covarianzas (varianzas y correlaciones), no los retornos esperados.

**4. A.** $y = 6.45/12.91 = 0.50$; $E(R_c) = 4.5 + 0.50 \times (9.44 - 4.5) = 6.97\%$. Equivalente: $4.5 + 0.382 \times 6.45$.

## 8. Lista de verificación

Sabes de esta semana si puedes: plantear el problema de Markowitz y decir qué insumos necesita; distinguir frontera de mínima varianza, frontera eficiente y portafolios dominados; explicar por qué el PMV no depende de $\mu$; calcular un ratio de Sharpe y un punto de la CAL; explicar por qué el tangente es el mismo para todos y qué decide la aversión al riesgo; y nombrar dos límites del modelo con su defensa práctica.

## 9. Mapa de repaso para el examen parcial (semanas 1 a 7)

| Sem. | Idea central | Fórmula clave | Nuestro ejemplo |
|---|---|---|---|
| 1 | Valor es flujo descontado | $VPN = -CF_0 + \sum CF_t/(1+r)^t$; TIR | VPN 5.41; TIR 12.71% |
| 2 | La tasa es un costo de oportunidad | $K_e = R_f + \beta\,ERP$; WACC con pesos de mercado | $K_e$ 11.1%; WACC 9.14% |
| 3 | Flujo y tasa consistentes | FCFF con WACC; FCFE con $K_e$ | FCFF 50.5; FCFE 46.75 |
| 4 | El valor terminal domina | $VT = FCFF_{n+1}/(WACC - g)$; puente EV a equity | 7.69 por acción |
| 5 | El mercado como contraste | P/E justificado; EV/EBITDA con puente | 6.75 por comparables |
| 6 | El riesgo no se promedia | $\sigma_p^2$ de dos activos; $\rho$; piso sistemático | 50/50: $\sigma_p$ 14.7% |
| 7 | Hay una mejor mezcla riesgosa | Sharpe; CAL; $y^{*}$ | $T$: Sharpe 0.382 |

Cada notebook de apuntes cierra con su lista de verificación y con las respuestas razonadas a los ítems tipo CFA de la clase: son el mejor simulacro. En la práctica de hoy hay un ejercicio de reforzamiento por semana.